# Qwen3.5 Audit v11 CVS Context v2 Baseline

Frozen old CVS baseline stays in `qwen3.5_audit_v11_seg_extract_spec_eval_with_cvs_context.ipynb`.
This notebook is the new `cvsctx_v2` workspace writing to its own baseline artifact directory.


In [1]:

from openai import OpenAI
import hashlib
import json
import os
import sys
import time
from pathlib import Path
from statistics import mean

try:
    import pandas as pd
except Exception:
    pd = None

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.action_segment_eval import (
    ACTORS,
    build_deterministic_structured_extraction,
    build_extraction_prompt as shared_build_extraction_prompt,
    build_messages_for_record,
    build_judge_prompt,
    build_simple_options,
    convert_record_to_simple_actions,
    default_segmentation_prompts,
    evaluate_method_predictions,
    extract_json_object,
    load_audit_records,
    naturalize_simple_actions,
    is_deterministic_structured_method,
    normalize_extraction,
    read_json,
    segmentation_source_method,
    write_json,
)

with open('/mnt/md0/weiqiuy/ips/carnaroli.txt') as input_file:
    carnaroli_ip = input_file.read().strip()
client = OpenAI(base_url=f'http://{carnaroli_ip}:8001/v1', api_key=os.environ['OPENAI_API_KEY'])

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B-FP8'
REQUEST_TEMPERATURE = 0
REQUEST_ENABLE_THINKING = False
ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'audit_v11'
FRAMES_DIR = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/frames/test'
SPEC_PATH = ANNOTATION_ROOT / 'specs/eval_spec.md'
ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v2__baseline'
SEGMENT_CACHE_PATH = ARTIFACT_DIR / 'segment_cache.json'
EXTRACTION_CACHE_PATH = ARTIFACT_DIR / 'extraction_cache.json'
JUDGE_CACHE_PATH = ARTIFACT_DIR / 'judge_cache.json'
RESULTS_PATH = ARTIFACT_DIR / 'results.json'
SPEC_EVAL_PATH = ARTIFACT_DIR / 'spec_eval_results.json'
JUDGE_SUMMARY_PATH = ARTIFACT_DIR / 'judge_summary.json'

FORCE_SEGMENT = False
FORCE_EXTRACTION = False
FORCE_JUDGE = False
DEBUG_RUN = False
DEBUG_RECORD_LIMIT = 5
MAX_RECORDS = None
WRITE_DERIVED_ARTIFACTS = True
BASE_METHODS = ['description_only', 'hint_questions', 'structured_prediction', 'structured_prediction_deterministic']
LABELING_VERSION = 'code_labels_v3_cvs_perframe_scores_v1'
METHODS = [f'{method}_cvs_context' for method in BASE_METHODS]

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('model:', MODEL_ID)
print('spec:', SPEC_PATH)
print('artifacts:', ARTIFACT_DIR)


model: Qwen/Qwen3.6-35B-A3B-FP8
spec: /mnt/md0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/specs/eval_spec.md
artifacts: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v2__baseline


In [2]:

records = load_audit_records(AUDIT_V11_DIR)
if DEBUG_RUN:
    records = records[:DEBUG_RECORD_LIMIT]
elif MAX_RECORDS:
    records = records[:MAX_RECORDS]

gt_simple_records = [convert_record_to_simple_actions(record) for record in records]
natural_gt_simple_records = [naturalize_simple_actions(record) for record in gt_simple_records]
simple_options = build_simple_options(gt_simple_records, ANNOTATION_ROOT)
if WRITE_DERIVED_ARTIFACTS or not (ARTIFACT_DIR / 'audit_v11_simple_actions.json').exists():
    write_json(ARTIFACT_DIR / 'audit_v11_simple_actions.json', gt_simple_records)
if WRITE_DERIVED_ARTIFACTS or not (ARTIFACT_DIR / 'audit_v11_simple_actions_natural_language.json').exists():
    write_json(ARTIFACT_DIR / 'audit_v11_simple_actions_natural_language.json', natural_gt_simple_records)
if WRITE_DERIVED_ARTIFACTS or not (ARTIFACT_DIR / 'audit_v11_simple_action_options.json').exists():
    write_json(ARTIFACT_DIR / 'audit_v11_simple_action_options.json', simple_options)


base_prompts = default_segmentation_prompts(simple_options)

CVS_CRITERION_DESCRIPTIONS = {
    'C1': 'Two and only two tubular structures are visible entering the gallbladder.',
    'C2': 'The hepatocystic triangle is cleared of fat and fibrous tissue.',
    'C3': 'The lower third of the gallbladder is detached from the liver bed.',
}

CVS_LABELS_DIR = Path('/mnt/md0/weiqiuy/datasets/CVS_Challenge_SAGES_v1/test/labels')
_cvs_score_cache: dict = {}


def load_cvs_frame_scores(video_id):
    """Return {frame_id: {'c1': k, 'c2': k, 'c3': k}} where k in {0,1,2,3} = #raters satisfied."""
    if video_id in _cvs_score_cache:
        return _cvs_score_cache[video_id]
    import csv
    path = CVS_LABELS_DIR / video_id / 'frame.csv'
    scores: dict = {}
    if path.exists():
        with open(path) as fh:
            for row in csv.DictReader(fh):
                fid = int(row['frame_id'])
                scores[fid] = {
                    'c1': sum(int(row[f'c1_rater{i}']) for i in (1, 2, 3)),
                    'c2': sum(int(row[f'c2_rater{i}']) for i in (1, 2, 3)),
                    'c3': sum(int(row[f'c3_rater{i}']) for i in (1, 2, 3)),
                }
    _cvs_score_cache[video_id] = scores
    return scores


def build_frame_labels_for_record(record):
    """Return {frame_id: label_str}; anchors get CVS scores, in-between frames omitted (default label used)."""
    scores = load_cvs_frame_scores(record['video_id'])
    labels = {}
    for fid, s in scores.items():
        labels[fid] = f"Frame {fid:06d}, C1={s['c1']}/3, C2={s['c2']}/3, C3={s['c3']}/3"
    return labels


def build_cvs_context_prompt(record, base_prompt):
    criterion = record.get('criterion')
    criterion_lines = '\n'.join(f'- {code}: {desc}' for code, desc in CVS_CRITERION_DESCRIPTIONS.items())
    coarse = record.get('coarse', {})
    context = f'''
CVS criteria definitions:
{criterion_lines}

Per-frame CVS satisfaction labels:
Anchor frames (every 5 seconds, i.e. at the stride-150 frame numbers) are annotated by 3 reviewers and shown with labels like "Frame 002100, C1=1/3, C2=2/3, C3=0/3". The value k/3 is the number of reviewers (of 3) who say that criterion is satisfied at that frame: 3/3 means all three reviewers agree it is satisfied, 0/3 means none, 1/3 means exactly one of three. The interleaved finer-grained frames (between anchors) carry only their frame number with no score because they were not separately rated.

This clip primarily concerns criterion {criterion}. Clip frame range: {coarse.get('start_frame')} to {coarse.get('end_frame')}.

Use the CVS context only as clip-level context. Still base segment boundaries and action timing on the provided frames and visible frame labels.
'''.strip()
    return context + '\n\n' + base_prompt


def prompt_for_record(record, method_name):
    base_method = segmentation_source_method(method_name).removesuffix('_cvs_context')
    return build_cvs_context_prompt(record, base_prompts[base_method])


def build_extraction_prompt(example_id, frame_range, method_name, prediction_text, options):
    prompt = shared_build_extraction_prompt(example_id, frame_range, method_name, prediction_text, options)
    left_rule = '''
Additional left-only rule for this v2 run:
- Use a left change code such as RETRACT_LATERAL_TO_MEDIAL or RETRACT_MEDIAL_TO_LATERAL only for the short segment where the retraction direction visibly transitions from one direction to another.
- Do not use a change code for a long interval that is merely described as holding the gallbladder upward and laterally, medially and upward, or otherwise stably exposed in multiple directions.
- If the segment is mostly stable rather than an actual transition, prefer the matching KEEP_RETRACT_* or RETRACT_* endpoint direction code instead of a *_TO_* code.
'''.strip()
    return prompt + '\n\n' + left_rule

prompts = {method: f'per-record CVS-context v2 baseline wrapper around {method.removesuffix("_cvs_context")}' for method in METHODS}
print('records:', len(records))
print('methods:', METHODS)
print('left options:', simple_options['left_retraction_direction_code_options'])
right_triplets = simple_options['right_triplet_options']
print('right tool_type options:', sorted({row[0] for row in right_triplets}))
print('right action_code options:', sorted({row[1] for row in right_triplets}))
print('right target_structure options:', sorted({row[2] for row in right_triplets}))
print('right target contexts:', simple_options['interface_options']['target_context'])
print('camera options:', simple_options['camera_action_code_options'])
print('other options:', simple_options['other_action_code_options'])


records: 90
methods: ['description_only_cvs_context', 'hint_questions_cvs_context', 'structured_prediction_cvs_context']
left options: ['KEEP_RETRACT_LATERAL', 'KEEP_RETRACT_MEDIAL', 'KEEP_RETRACT_UPWARD', 'RETRACT_LATERAL', 'RETRACT_LATERAL_TO_MEDIAL', 'RETRACT_LATERAL_TO_UPWARD', 'RETRACT_MEDIAL', 'RETRACT_MEDIAL_TO_LATERAL', 'RETRACT_UPWARD_TO_LATERAL']
right tool_type options: ['Hook', 'Irrigator', 'Maryland', 'Scissors', 'clipper']
right action_code options: ['CLIP', 'COAGULATE_HEMOSTASIS', 'COUNTERTRACTION_ASSIST', 'DISSECT', 'IRRIGATOR_ASPIRATE', 'RETRACT_DOWNWARD', 'TOOL_WITHDRAW_UNBLOCKS_VIEW', 'sweeping']
right target_structure options: ['CysticArtery', 'CysticDuct', 'CysticPlate', 'GallbladderNeck_Infundibulum', 'HepatocysticTriangle']
right target contexts: ['between presumed cystic duct and presumed cystic artery', 'between cystic artery and cystic plate', 'near the base of the hepatocystic triangle', 'between cystic artery and liver bed', 'moving from a posterior view to 

In [3]:

def cache_get(path, default=None):
    return read_json(path, default if default is not None else {})


def cache_set(path, obj):
    write_json(path, obj)


def model_json_call(messages, cache_path, cache_key, signature, *, force=False, repair_label=None):
    cache = cache_get(cache_path, {})
    entry = cache.get(cache_key)
    if force or not entry or entry.get('signature') != signature:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=REQUEST_TEMPERATURE,
            extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
        )
        answer_raw = response.choices[0].message.content
        try:
            answer_json = extract_json_object(answer_raw)
            parse_error = None
        except Exception as exc:
            parse_error = repr(exc)
            malformed_path = ARTIFACT_DIR / f'malformed_{repair_label or "json"}_{cache_key.replace("/", "_").replace("::", "__")}.txt'
            malformed_path.write_text(answer_raw)
            repair_messages = [
                {'role': 'system', 'content': 'Repair malformed JSON. Return valid JSON only.'},
                {'role': 'user', 'content': f'Parse error: {parse_error}\n\nMalformed output:\n{answer_raw}'},
            ]
            repair_response = client.chat.completions.create(
                model=MODEL_ID,
                messages=repair_messages,
                temperature=0,
                extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
            )
            repaired_raw = repair_response.choices[0].message.content
            (ARTIFACT_DIR / f'repaired_{repair_label or "json"}_{cache_key.replace("/", "_").replace("::", "__")}.txt').write_text(repaired_raw)
            answer_json = extract_json_object(repaired_raw)
        entry = {'answer_raw': answer_raw, 'answer_json': answer_json, 'signature': signature, 'parse_error_repaired': parse_error}
        cache[cache_key] = entry
        cache_set(cache_path, cache)
    return entry


def run_segmentation(record, method_name):
    messages, prompt_record = build_messages_for_record(
        record,
        prompt_for_record(record, method_name),
        FRAMES_DIR,
        MODEL_ID,
        REQUEST_TEMPERATURE,
        REQUEST_ENABLE_THINKING,
        frame_labels=build_frame_labels_for_record(record),
    )
    prompt_method = segmentation_source_method(method_name)
    prompt_record['method'] = prompt_method
    signature = hashlib.sha256(json.dumps(prompt_record, sort_keys=True).encode('utf-8')).hexdigest()
    cache_key = f'{record["example_id"]}::{prompt_method}'
    cache = cache_get(SEGMENT_CACHE_PATH, {})
    entry = cache.get(cache_key)
    if FORCE_SEGMENT or not entry or entry.get('signature') != signature:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=REQUEST_TEMPERATURE,
            extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
        )
        entry = {'answer_raw': response.choices[0].message.content, 'prompt_record': prompt_record, 'signature': signature}
        cache[cache_key] = entry
        cache_set(SEGMENT_CACHE_PATH, cache)
    return entry


def run_extraction(record, method_name, segmentation_entry):
    frame_range = [int(record['coarse']['start_frame']), int(record['coarse']['end_frame'])]
    cache_key = f'{record["example_id"]}::{method_name}'
    if is_deterministic_structured_method(method_name):
        answer_json = build_deterministic_structured_extraction(method_name, segmentation_entry['answer_raw'], frame_range[0], frame_range[1])
        signature = hashlib.sha256(json.dumps({'run_id': 'structured_prediction_deterministic_extraction_v1_cvsctx_v2_baseline', 'segmentation_signature': segmentation_entry.get('signature'), 'answer_json': answer_json}, sort_keys=True).encode('utf-8')).hexdigest()
        cache = cache_get(EXTRACTION_CACHE_PATH, {})
        entry = cache.get(cache_key)
        if FORCE_EXTRACTION or not entry or entry.get('signature') != signature:
            entry = {'answer_raw': json.dumps(answer_json, indent=2), 'answer_json': answer_json, 'signature': signature}
    else:
        prompt = build_extraction_prompt(record['example_id'], frame_range, method_name, segmentation_entry['answer_raw'], simple_options)
        signature = hashlib.sha256(json.dumps({'run_id': 'taxonomy_extraction_v1_cvsctx_v2_baseline', 'model': MODEL_ID, 'prompt': prompt}, sort_keys=True).encode('utf-8')).hexdigest()
        entry = model_json_call(
            [
                {'role': 'system', 'content': 'Extract constrained timestamped surgical actions. Return valid JSON only.'},
                {'role': 'user', 'content': prompt},
            ],
            EXTRACTION_CACHE_PATH,
            cache_key,
            signature,
            force=FORCE_EXTRACTION,
            repair_label='extraction',
        )
    entry['normalized'] = normalize_extraction(entry['answer_json'], frame_range[0], frame_range[1], simple_options)
    entry['normalized_natural_language'] = naturalize_simple_actions(entry['normalized'])
    cache = cache_get(EXTRACTION_CACHE_PATH, {})
    cache[cache_key] = entry
    cache_set(EXTRACTION_CACHE_PATH, cache)
    return entry


def run_llm_judge(record, method_name, candidate, candidate_kind):
    gt_simple = next(item for item in gt_simple_records if item['example_id'] == record['example_id'])
    frame_range = gt_simple['frame_range']
    prompt = build_judge_prompt(record['example_id'], frame_range, gt_simple, candidate, method_name, candidate_kind)
    signature = hashlib.sha256(json.dumps({'run_id': 'rationale_then_score_judge_code_labels_v3_cvsctx_v2_baseline', 'model': MODEL_ID, 'prompt': prompt}, sort_keys=True).encode('utf-8')).hexdigest()
    cache_key = f'{record["example_id"]}::{method_name}::{candidate_kind}'
    return model_json_call(
        [
            {'role': 'system', 'content': 'You are a strict evaluator. For each rubric item, write rationale first, then score. Return valid JSON only.'},
            {'role': 'user', 'content': prompt},
        ],
        JUDGE_CACHE_PATH,
        cache_key,
        signature,
        force=FORCE_JUDGE,
        repair_label='judge',
    )


In [4]:

results_payload = cache_get(RESULTS_PATH, {'results': []})
results_by_key = {(r.get('example_id'), r.get('method')): r for r in results_payload.get('results', [])}
start_time = time.time()
total = len(records) * len(METHODS)
step = 0

for record in records:
    for method_name in METHODS:
        step += 1
        key = (record['example_id'], method_name)
        existing = results_by_key.get(key)
        if existing and existing.get('status') == 'ok' and existing.get('labeling_version') == LABELING_VERSION and not (FORCE_SEGMENT or FORCE_EXTRACTION or FORCE_JUDGE):
            if step % 10 == 0 or step == total:
                print(f'[{step}/{total}] cached {method_name} {record["example_id"]}')
            continue
        try:
            seg_entry = run_segmentation(record, method_name)
            extraction_entry = run_extraction(record, method_name, seg_entry)
            original_judge = run_llm_judge(record, method_name, seg_entry['answer_raw'], 'original_text')
            extracted_judge = run_llm_judge(record, method_name, extraction_entry['normalized'], 'extracted_json')
            item = {
                'status': 'ok',
                'labeling_version': LABELING_VERSION,
                'example_id': record['example_id'],
                'video_id': record['video_id'],
                'criterion': record.get('criterion'),
                'method': method_name,
                'segmentation_answer': seg_entry['answer_raw'],
                'extracted_actions': extraction_entry['normalized'],
                'extracted_actions_natural_language': extraction_entry['normalized_natural_language'],
                'original_llm_judge': original_judge['answer_json'],
                'extracted_llm_judge': extracted_judge['answer_json'],
            }
        except Exception as exc:
            item = {
                'status': 'error',
                'labeling_version': LABELING_VERSION,
                'example_id': record.get('example_id'),
                'video_id': record.get('video_id'),
                'criterion': record.get('criterion'),
                'method': method_name,
                'error': repr(exc),
            }
            print('ERROR', item['example_id'], method_name, repr(exc))
        results_by_key[key] = item
        cache_set(RESULTS_PATH, {'results': list(results_by_key.values())})
        print(f'[{step}/{total}] {item["status"]} {method_name} {record["example_id"]}')

elapsed = (time.time() - start_time) / 60
if WRITE_DERIVED_ARTIFACTS:
    cache_set(RESULTS_PATH, {'results': list(results_by_key.values())})
print(f'done/paused after {elapsed:.1f} min')
print('ok:', sum(1 for r in results_by_key.values() if r.get('status') == 'ok'), 'errors:', sum(1 for r in results_by_key.values() if r.get('status') == 'error'))


[10/270] cached description_only_cvs_context 00467596-8200-449c-8528-d4816ec2f6a2__C2__avg__c_001350_001650
[16/270] ok description_only_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C1__avg__c_001650_001950
[17/270] ok hint_questions_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C1__avg__c_001650_001950
[18/270] ok structured_prediction_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C1__avg__c_001650_001950
[19/270] ok description_only_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C2__avg__c_000450_001050
[20/270] ok hint_questions_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C2__avg__c_000450_001050
[21/270] ok structured_prediction_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C2__avg__c_000450_001050
[22/270] ok description_only_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C3__avg__c_001350_002100
[23/270] ok hint_questions_cvs_context 004fa282-7e76-473e-bf34-01fe831368dc__C3__avg__c_001350_002100
[24/270] ok structured_prediction_cvs_context 004fa282-7

In [7]:
from IPython.display import HTML, display

results = cache_get(RESULTS_PATH, {'results': []})['results']
ok_results = [item for item in results if item.get('status') == 'ok']

spec_eval = {}
for method_name in METHODS:
    pred_records = []
    for item in ok_results:
        if item['method'] != method_name:
            continue
        pred = dict(item['extracted_actions'])
        pred['example_id'] = item['example_id']
        pred['video_id'] = item['video_id']
        pred['criterion'] = item.get('criterion')
        pred_records.append(pred)
    spec_rows = evaluate_method_predictions(pred_records, gt_simple_records)
    spec_eval[method_name] = spec_rows

if WRITE_DERIVED_ARTIFACTS:
    cache_set(SPEC_EVAL_PATH, spec_eval)
flat_rows = []
for method_name, rows in spec_eval.items():
    for row in rows:
        flat_rows.append({k: v for k, v in {'method': method_name, **row}.items() if k != 'details'})


def hide_index(styler):
    try:
        return styler.hide(axis='index')
    except Exception:
        return styler.hide_index()


def style_best_by_row(styler, score_columns, precision=3):
    def bold_best(row):
        vals = pd.to_numeric(row[score_columns], errors='coerce')
        if vals.isna().all():
            return [''] * len(row)
        best = vals.max()
        return [
            'font-weight: 700; background-color: #e7f3ec' if col in score_columns and pd.notna(row[col]) and row[col] == best else ''
            for col in row.index
        ]

    return (
        styler
        .format({col: f'{{:.{precision}f}}' for col in score_columns}, na_rep='')
        .apply(bold_best, axis=1)
        .set_properties(subset=score_columns, **{'text-align': 'right'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]},
            {'selector': 'td', 'props': [('padding', '4px 8px')]},
        ])
    )


import html as _html


def render_fast_table(df, score_columns=(), precision=3, formatters=None, numeric_columns=None, bold_best=True):
    score_columns = list(score_columns)
    formatters = formatters or {}
    numeric_columns = set(numeric_columns or score_columns)
    styles = '\n<style>\n.fast-eval-table { border-collapse: collapse; margin: 0.5em 0 1.2em 0; font-size: 13px; }\n.fast-eval-table th { text-align: left; padding: 4px 8px; border-bottom: 1px solid #bbb; }\n.fast-eval-table td { padding: 4px 8px; border-bottom: 1px solid #eee; vertical-align: top; }\n.fast-eval-table td.num { text-align: right; font-variant-numeric: tabular-nums; }\n.fast-eval-table td.best { font-weight: 700; background-color: #e7f3ec; }\n</style>\n'

    def format_value(col, val):
        try:
            if pd.isna(val):
                return ''
        except Exception:
            pass
        if col in formatters:
            return formatters[col](val)
        if col in score_columns:
            try:
                return f'{float(val):.{precision}f}'
            except Exception:
                return str(val)
        return str(val)

    header = ''.join(f'<th>{_html.escape(str(col))}</th>' for col in df.columns)
    body_rows = []
    for _, row in df.iterrows():
        best = None
        if bold_best and score_columns:
            vals = pd.to_numeric(row[score_columns], errors='coerce')
            if not vals.isna().all():
                best = vals.max()
        cells = []
        for col in df.columns:
            val = row[col]
            classes = []
            if col in numeric_columns:
                classes.append('num')
            if col in score_columns and best is not None:
                try:
                    if pd.notna(val) and float(val) == float(best):
                        classes.append('best')
                except Exception:
                    pass
            class_attr = f' class="{" ".join(classes)}"' if classes else ''
            cells.append(f'<td{class_attr}>{_html.escape(format_value(col, val))}</td>')
        body_rows.append('<tr>' + ''.join(cells) + '</tr>')
    return styles + '<table class="fast-eval-table"><thead><tr>' + header + '</tr></thead><tbody>' + ''.join(body_rows) + '</tbody></table>'

if pd is not None and flat_rows:
    spec_df = pd.DataFrame(flat_rows)
    if WRITE_DERIVED_ARTIFACTS:
        spec_df.to_csv(ARTIFACT_DIR / 'spec_eval_summary.csv', index=False)
    method_columns = [method for method in METHODS if method in set(spec_df['method'])]

    spec_comparison = (
        spec_df
        .pivot_table(
            index=['actor', 'granularity', 'iou'],
            columns='method',
            values='f1_mean',
            aggfunc='mean',
        )
        .reset_index()
        .sort_values(['actor', 'granularity', 'iou'])
    )
    for method in method_columns:
        if method not in spec_comparison.columns:
            spec_comparison[method] = None
    spec_comparison = spec_comparison[['actor', 'granularity', 'iou', *method_columns]]
    if WRITE_DERIVED_ARTIFACTS:
        spec_comparison.to_csv(ARTIFACT_DIR / 'spec_f1_comparison_table.csv', index=False)

    actor_aggregate = (
        spec_df
        .groupby(['actor', 'method'], as_index=False)
        .agg(mean_f1=('f1_mean', 'mean'), mean_std=('f1_std', 'mean'), rows=('f1_mean', 'size'))
        .pivot_table(index='actor', columns='method', values='mean_f1', aggfunc='mean')
        .reset_index()
        .sort_values('actor')
    )
    for method in method_columns:
        if method not in actor_aggregate.columns:
            actor_aggregate[method] = None
    actor_aggregate = actor_aggregate[['actor', *method_columns]]
    if WRITE_DERIVED_ARTIFACTS:
        actor_aggregate.to_csv(ARTIFACT_DIR / 'spec_f1_actor_aggregate_table.csv', index=False)

    overall_aggregate = (
        spec_df
        .groupby('method', as_index=False)
        .agg(mean_f1=('f1_mean', 'mean'), rows=('f1_mean', 'size'))
        .set_index('method')
        .reindex(method_columns)
        .reset_index()
        .rename(columns={'method': 'method', 'mean_f1': 'aggregate mean F1'})
    )
    overall_display = overall_aggregate[['method', 'aggregate mean F1', 'rows']]
    if WRITE_DERIVED_ARTIFACTS:
        overall_display.to_csv(ARTIFACT_DIR / 'spec_f1_overall_aggregate.csv', index=False)

    display(HTML('<h3>Spec eval aggregate F1</h3>'))
    display(HTML(render_fast_table(
        overall_display,
        score_columns=['aggregate mean F1'],
        numeric_columns=['aggregate mean F1', 'rows'],
        formatters={'aggregate mean F1': lambda v: f'{float(v):.3f}', 'rows': lambda v: f'{float(v):.0f}'},
        bold_best=False,
    )))
    display(HTML('<h3>Spec eval F1 by actor</h3><p>Best method per row is bolded.</p>'))
    display(HTML(render_fast_table(actor_aggregate, method_columns)))
    display(HTML('<h3>Spec eval F1 by actor, granularity, and IoU</h3><p>Best method per row is bolded.</p>'))
    display(HTML(render_fast_table(spec_comparison, method_columns)))
else:
    print(json.dumps(flat_rows, indent=2))


method,aggregate mean F1,rows
description_only_cvs_context,0.215,30
hint_questions_cvs_context,0.269,30
structured_prediction_cvs_context,0.326,30


actor,description_only_cvs_context,hint_questions_cvs_context,structured_prediction_cvs_context
camera,0.029,0.135,0.182
left,0.147,0.416,0.576
other,1.000,0.417,0.417
right,0.207,0.206,0.189


actor,granularity,iou,description_only_cvs_context,hint_questions_cvs_context,structured_prediction_cvs_context
camera,coarse,0.1,0.072,0.241,0.334
camera,coarse,0.25,0.043,0.154,0.261
camera,coarse,0.5,0.000,0.099,0.190
camera,exact,0.1,0.048,0.176,0.167
camera,exact,0.25,0.024,0.101,0.110
camera,exact,0.5,0.000,0.071,0.077
camera,medium,0.1,0.048,0.182,0.216
camera,medium,0.25,0.024,0.113,0.159
camera,medium,0.5,0.000,0.080,0.126
left,coarse,0.1,0.250,0.546,0.568


In [8]:
from IPython.display import HTML, display


def summarize_judge(results, judge_key):
    rows = []
    for item in results:
        if item.get('status') != 'ok':
            continue
        judge = item.get(judge_key, {})
        for rubric in judge.get('rubric', []):
            try:
                score = float(rubric.get('score'))
            except Exception:
                continue
            rows.append({
                'method': item['method'],
                'candidate_kind': judge.get('candidate_kind', judge_key),
                'item': rubric.get('item'),
                'score': score,
            })
    summary = []
    groups = sorted({(r['method'], r['candidate_kind'], r['item']) for r in rows})
    for method, candidate_kind, rubric_item in groups:
        vals = [r['score'] for r in rows if (r['method'], r['candidate_kind'], r['item']) == (method, candidate_kind, rubric_item)]
        summary.append({'method': method, 'candidate_kind': candidate_kind, 'rubric_item': rubric_item, 'n': len(vals), 'mean_score': mean(vals) if vals else None})
    return summary


def rubric_group(rubric_item):
    actor = str(rubric_item).split('_', 1)[0]
    return {
        'left': 'Left hand retraction',
        'right': 'Right hand dissection',
        'camera': 'Camera movement',
        'other': 'Other actions',
    }.get(actor, 'Other rubric items')


def rubric_sort_key(rubric_item):
    text = str(rubric_item)
    actor, _, detail = text.partition('_')
    actor_order = {'left': 0, 'right': 1, 'camera': 2, 'other': 3}
    detail_order = {'action': 0, 'tool': 1, 'target': 2, 'context': 3, 'time': 4}
    return (actor_order.get(actor, 99), detail_order.get(detail, 99), text)


def candidate_label(candidate_kind):
    return {
        'original_text': 'Original segmentation text',
        'extracted_json': 'Extracted code JSON',
    }.get(candidate_kind, candidate_kind)


def hide_index(styler):
    try:
        return styler.hide(axis='index')
    except Exception:
        return styler.hide_index()


def style_best_by_row(styler, score_columns, precision=3):
    def bold_best(row):
        vals = pd.to_numeric(row[score_columns], errors='coerce')
        if vals.isna().all():
            return [''] * len(row)
        best = vals.max()
        return [
            'font-weight: 700; background-color: #e7f3ec' if col in score_columns and pd.notna(row[col]) and row[col] == best else ''
            for col in row.index
        ]

    return (
        styler
        .format({col: f'{{:.{precision}f}}' for col in score_columns}, na_rep='')
        .apply(bold_best, axis=1)
        .set_properties(subset=score_columns, **{'text-align': 'right'})
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]},
            {'selector': 'td', 'props': [('padding', '4px 8px')]},
        ])
    )



import html as _html


def render_fast_table(df, score_columns=(), precision=3, formatters=None, numeric_columns=None, bold_best=True):
    score_columns = list(score_columns)
    formatters = formatters or {}
    numeric_columns = set(numeric_columns or score_columns)
    styles = '\n<style>\n.fast-eval-table { border-collapse: collapse; margin: 0.5em 0 1.2em 0; font-size: 13px; }\n.fast-eval-table th { text-align: left; padding: 4px 8px; border-bottom: 1px solid #bbb; }\n.fast-eval-table td { padding: 4px 8px; border-bottom: 1px solid #eee; vertical-align: top; }\n.fast-eval-table td.num { text-align: right; font-variant-numeric: tabular-nums; }\n.fast-eval-table td.best { font-weight: 700; background-color: #e7f3ec; }\n</style>\n'

    def format_value(col, val):
        try:
            if pd.isna(val):
                return ''
        except Exception:
            pass
        if col in formatters:
            return formatters[col](val)
        if col in score_columns:
            try:
                return f'{float(val):.{precision}f}'
            except Exception:
                return str(val)
        return str(val)

    header = ''.join(f'<th>{_html.escape(str(col))}</th>' for col in df.columns)
    body_rows = []
    for _, row in df.iterrows():
        best = None
        if bold_best and score_columns:
            vals = pd.to_numeric(row[score_columns], errors='coerce')
            if not vals.isna().all():
                best = vals.max()
        cells = []
        for col in df.columns:
            val = row[col]
            classes = []
            if col in numeric_columns:
                classes.append('num')
            if col in score_columns and best is not None:
                try:
                    if pd.notna(val) and float(val) == float(best):
                        classes.append('best')
                except Exception:
                    pass
            class_attr = f' class="{" ".join(classes)}"' if classes else ''
            cells.append(f'<td{class_attr}>{_html.escape(format_value(col, val))}</td>')
        body_rows.append('<tr>' + ''.join(cells) + '</tr>')
    return styles + '<table class="fast-eval-table"><thead><tr>' + header + '</tr></thead><tbody>' + ''.join(body_rows) + '</tbody></table>'

judge_summary = {
    'original_text': summarize_judge(ok_results, 'original_llm_judge'),
    'extracted_json': summarize_judge(ok_results, 'extracted_llm_judge'),
}
if WRITE_DERIVED_ARTIFACTS:
    cache_set(JUDGE_SUMMARY_PATH, judge_summary)
flat = judge_summary['original_text'] + judge_summary['extracted_json']
if pd is not None and flat:
    judge_df = pd.DataFrame(flat)
    if WRITE_DERIVED_ARTIFACTS:
        judge_df.to_csv(ARTIFACT_DIR / 'llm_judge_summary.csv', index=False)

    judge_df['rubric_group'] = judge_df['rubric_item'].map(rubric_group)
    judge_df['rubric_order'] = judge_df['rubric_item'].map(rubric_sort_key)
    judge_df['candidate_display'] = judge_df['candidate_kind'].map(candidate_label)
    method_columns = [method for method in METHODS if method in set(judge_df['method'])]

    comparison = (
        judge_df
        .pivot_table(
            index=['candidate_kind', 'candidate_display', 'rubric_group', 'rubric_order', 'rubric_item'],
            columns='method',
            values='mean_score',
            aggfunc='mean',
        )
        .reset_index()
        .sort_values(['candidate_kind', 'rubric_order'])
        .drop(columns=['candidate_kind', 'rubric_order'])
        .rename(columns={'candidate_display': 'candidate', 'rubric_group': 'rubric group', 'rubric_item': 'rubric item'})
    )
    for method in method_columns:
        if method not in comparison.columns:
            comparison[method] = None
    comparison = comparison[['candidate', 'rubric group', 'rubric item', *method_columns]]
    if WRITE_DERIVED_ARTIFACTS:
        comparison.to_csv(ARTIFACT_DIR / 'llm_judge_comparison_table.csv', index=False)

    aggregate = (
        judge_df
        .groupby(['candidate_kind', 'candidate_display', 'method'], as_index=False)
        .agg(mean_score=('mean_score', 'mean'), rubric_items=('rubric_item', 'nunique'), n_scores=('n', 'sum'))
    )
    aggregate_table = (
        aggregate
        .pivot_table(index=['candidate_kind', 'candidate_display'], columns='method', values='mean_score', aggfunc='mean')
        .reset_index()
        .sort_values('candidate_kind')
        .drop(columns=['candidate_kind'])
        .rename(columns={'candidate_display': 'candidate'})
    )
    for method in method_columns:
        if method not in aggregate_table.columns:
            aggregate_table[method] = None
    aggregate_table = aggregate_table[['candidate', *method_columns]]
    if WRITE_DERIVED_ARTIFACTS:
        aggregate_table.to_csv(ARTIFACT_DIR / 'llm_judge_aggregate_table.csv', index=False)

    display(HTML('<h3>LLM judge aggregate mean score</h3><p>Best method per row is bolded.</p>'))
    display(HTML(render_fast_table(aggregate_table, method_columns)))
    display(HTML('<h3>LLM judge rubric comparison</h3><p>Rubric items are grouped by related surgical dimension. Best method per row is bolded.</p>'))
    display(HTML(render_fast_table(comparison, method_columns)))
else:
    print(json.dumps(judge_summary, indent=2))


candidate,description_only_cvs_context,hint_questions_cvs_context,structured_prediction_cvs_context
Extracted code JSON,0.481,0.485,0.492
Original segmentation text,0.570,0.599,0.509


candidate,rubric group,rubric item,description_only_cvs_context,hint_questions_cvs_context,structured_prediction_cvs_context
Extracted code JSON,Left hand retraction,left_action,0.144,0.267,0.450
Extracted code JSON,Left hand retraction,left_time,0.161,0.522,0.511
Extracted code JSON,Right hand dissection,right_action,0.667,0.711,0.606
Extracted code JSON,Right hand dissection,right_tool,0.550,0.689,0.567
Extracted code JSON,Right hand dissection,right_target,0.417,0.383,0.372
Extracted code JSON,Right hand dissection,right_context,0.167,0.217,0.133
Extracted code JSON,Right hand dissection,right_time,0.300,0.311,0.300
Extracted code JSON,Camera movement,camera_action,0.461,0.150,0.256
Extracted code JSON,Camera movement,camera_time,0.439,0.139,0.267
Extracted code JSON,Other actions,other_action,1.000,0.978,0.978


In [9]:
1

1